In [ ]:
# transformers がすでにインストールされているかを確認し、
# 未インストールの場合のみインストールする
import importlib.util
import sys
import subprocess

# importlib.util.find_spec("transformers") は、
# Python環境内に transformers パッケージが存在するかを確認する
# None の場合は未インストール、None でなければインストール済み
if importlib.util.find_spec("transformers") is None:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "transformers[ja,sentencepiece,torch]"]
    )
else:
    print("transformers はすでにインストールされています。")


In [3]:
import os
import warnings

# Hugging Face Hub からの認証なしリクエストに関する警告など、
# 実行結果に不要な警告を表示しないようにする。
warnings.filterwarnings("ignore")

# Hugging Face Hub の進捗バーを無効化する。
# モデルの重みダウンロード時に表示される tqdm の progress bar を抑制する。
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"

# Transformers 側のログ出力を error のみに制限する。
# これにより、モデル読み込み時の warning / info レベルの出力を抑える。
from transformers.utils import logging

logging.set_verbosity_error()

from transformers import pipeline

# Hugging Face Transformers の pipeline を使って、
# 事前学習済み・ファインチューニング済みモデルによるテキスト分類を行う。
#
# pipeline は、トークナイズ・モデルへの入力・推論・出力整形をまとめて行う高水準API。
# ここでは感情極性分類、つまり文章がポジティブかネガティブかを判定するモデルを使う。
text_classification_pipeline = pipeline(
    # text-classification を明示することで、
    # この pipeline が文章分類タスク用であることをコード上から分かりやすくする。
    "text-classification",
    # llm-book/bert-base-japanese-v3-marc_ja は、
    # 日本語BERTを日本語レビュー分類データセットでファインチューニングしたモデル。
    #
    # BERTはTransformer Encoderをベースにしたモデルであり、
    # 文中の各トークンがSelf-Attentionによって他のトークンとの関係を考慮しながら表現される。
    #
    # テキスト分類では、文章全体の意味を表すベクトルを分類層に入力し、
    # positive / negative などのラベルごとのスコアを計算する。
    model="llm-book/bert-base-japanese-v3-marc_ja",
)

# ポジティブ寄りの文章を用意する。
# 「感動する」という語が含まれており、文全体として肯定的な評価を表している。
positive_text = "世界には言葉がわからなくても感動する音楽がある。"

# ネガティブ寄りの文章を用意する。
# 「ひどい」という語が含まれており、文全体として否定的な評価を表している。
negative_text = "世界には言葉がでないほどひどい音楽がある。"

# 複数の文章をリストにまとめて pipeline に渡す。
#
# 1文ずつ推論することもできるが、複数文をまとめて渡すことで、
# 入力と出力の対応関係を管理しやすくなる。
texts = [
    positive_text,
    negative_text,
]

# texts の各文章について感情極性を予測する。
#
# 内部では以下の処理が行われる。
#
# 1. 文章をトークンに分割する
# 2. トークンをID列に変換する
# 3. BERTに入力してSelf-Attentionにより文脈表現を得る
# 4. 分類層で各ラベルのスコアを計算する
# 5. 最も確率の高いラベルとそのスコアを返す
results = text_classification_pipeline(texts)

# 余計な説明文を出さず、分類結果だけを表示する。
for result in results:
    print(result)


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 20540.78it/s]


{'label': 'positive', 'score': 0.9993619322776794}
{'label': 'negative', 'score': 0.9636250138282776}


In [4]:
from transformers import pipeline

# Hugging Face Transformers の pipeline を使って、
# 自然言語推論 Natural Language Inference, NLI を行う。
#
# NLI は、2つの文の論理的な関係を分類するタスク。
# 主に以下の3種類のラベルに分類される。
#
# entailment:
#   1つ目の文が正しいとき、2つ目の文も必ず正しいと考えられる関係。
#   日本語では「含意」と呼ばれる。
#
# contradiction:
#   1つ目の文が正しいとき、2つ目の文は成り立たない関係。
#   日本語では「矛盾」と呼ばれる。
#
# neutral:
#   1つ目の文だけでは、2つ目の文が正しいとも間違いとも判断できない関係。
#   日本語では「中立」と呼ばれる。
#
# 例:
#   前提文: 「人が犬を散歩させている」
#   仮説文: 「人が動物と一緒にいる」
#   この場合、犬は動物なので entailment になりやすい。
#
#   前提文: 「人が犬を散歩させている」
#   仮説文: 「人が一人で歩いている」
#   この場合、犬と一緒にいるため contradiction になりやすい。
#
#   前提文: 「人が犬を散歩させている」
#   仮説文: 「その人は公園にいる」
#   この場合、公園かどうかは前提文だけでは分からないため neutral になりやすい。
nli_pipeline = pipeline(
    # llm-book/bert-base-japanese-v3-jnli は、
    # 日本語BERTを日本語自然言語推論データセット JNLI で
    # ファインチューニングしたモデル。
    #
    # BERT は Transformer Encoder に基づくモデルで、
    # Self-Attention によって文中の各トークンが他のトークンを参照しながら
    # 文脈を考慮した表現を獲得する。
    #
    # NLIでは、2つの文をまとめてBERTに入力する。
    # 一般にBERTでは、文ペアを以下のような形で扱う。
    #
    #   [CLS] 文1 [SEP] 文2 [SEP]
    #
    # [CLS] トークンに対応する出力ベクトルを文ペア全体の表現として使い、
    # その表現を分類層に入力して entailment / contradiction / neutral を予測する。
    model="llm-book/bert-base-japanese-v3-jnli"
)

# 前提文 premise を用意する。
#
# NLIでは、1つ目の文を「前提文」と呼ぶ。
# この文が事実として成り立っていると仮定したとき、
# 2つ目の文が論理的にどういう関係にあるかを判定する。
text = "二人の男性がジェット機を見ています"

# 含意関係になりやすい仮説文 hypothesis を用意する。
#
# 前提文:
#   「二人の男性がジェット機を見ています」
#
# 仮説文:
#   「ジェット機を見ている人が二人います」
#
# 前提文が正しければ、
# 「ジェット機を見ている人が二人いる」ことも自然に成り立つ。
# そのため、この文ペアは entailment と判定されることが期待される。
entailment_text = "ジェット機を見ている人が二人います"

# text と entailment_text の論理関係を予測する。
#
# pipeline に文ペアを入力するときは、
# {"text": 前提文, "text_pair": 仮説文} の辞書形式で渡す。
#
# 内部では、BERTのTokenizerが2文をまとめてトークン化し、
# token_type_ids によって「どこまでが1文目で、どこからが2文目か」を区別する。
#
# その後、BERTが文ペア全体の意味関係をSelf-Attentionで捉え、
# 最終的に3クラス分類としてラベルとスコアを返す。
print(nli_pipeline({"text": text, "text_pair": entailment_text}))

# 矛盾関係になりやすい仮説文を用意する。
#
# 前提文:
#   「二人の男性がジェット機を見ています」
#
# 仮説文:
#   「二人の男性が飛んでいます」
#
# 前提文では、男性たちは「ジェット機を見ている」と述べられている。
# 一方で、仮説文では男性たち自身が「飛んでいる」と述べられている。
#
# 通常の解釈では、
# 「ジェット機を見ている男性」と「男性自身が飛んでいる」は同時に成立しにくい。
# そのため、この文ペアは contradiction と判定されることが期待される。
contradiction_text = "二人の男性が飛んでいます"

# text と contradiction_text の論理関係を予測する。
#
# NLIモデルは、単語の一致だけではなく、
# 「見る」と「飛ぶ」の意味的な違いや、
# 主体が誰であるかを文脈的に考慮して分類する。
#
# ただし、モデルはあくまで学習データから得た統計的パターンに基づいて予測するため、
# 人間の論理推論と完全に一致するとは限らない。
print(nli_pipeline({"text": text, "text_pair": contradiction_text}))

# 中立関係になりやすい仮説文を用意する。
#
# 前提文:
#   「二人の男性がジェット機を見ています」
#
# 仮説文:
#   「2人の男性が、白い飛行機を眺めています」
#
# 「2人の男性が飛行機を見ている」という点は前提文と近い。
# しかし、前提文ではジェット機の色が「白い」とは述べられていない。
#
# そのため、前提文だけから
# 「白い飛行機である」とまでは断定できない。
# このように、正しいとも間違いとも言い切れない場合は neutral になりやすい。
neutral_text = "2人の男性が、白い飛行機を眺めています"

# text と neutral_text の論理関係を予測する。
#
# この例では、かなり意味が近い文同士だが、
# 「白い」という追加情報が前提文にないため、
# 厳密な自然言語推論では entailment ではなく neutral とみなされる可能性がある。
#
# NLIでは、このような「前提文に書かれていない追加情報」を
# どのように扱うかが重要になる。
print(nli_pipeline({"text": text, "text_pair": neutral_text}))


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 24261.97it/s]


{'label': 'entailment', 'score': 0.9958741068840027}
{'label': 'contradiction', 'score': 0.9922866821289062}
{'label': 'neutral', 'score': 0.9821659922599792}


In [5]:
from transformers import pipeline

# Hugging Face Transformers の pipeline を使って、
# 2つの日本語文の意味的類似度を計算する。
#
# ここで行うタスクは STS Semantic Textual Similarity と呼ばれる。
# STS は、2つの文が意味的にどれくらい近いかを数値で評価するタスク。
#
# 例えば、
#   「犬が走っている」
#   「動物が走っている」
# は意味が近いため高い類似度になりやすい。
#
# 一方で、
#   「犬が走っている」
#   「机の上に本がある」
# は意味がかなり異なるため低い類似度になりやすい。
#
# NLI が entailment / contradiction / neutral のような離散的な分類を行うのに対して、
# STS は「どれくらい似ているか」を連続値として出力する点が異なる。
text_sim_pipeline = pipeline(
    # llm-book/bert-base-japanese-v3-jsts は、
    # 日本語BERTを日本語文類似度データセット JSTS で
    # ファインチューニングしたモデル。
    #
    # JSTS は Japanese Semantic Textual Similarity の略で、
    # 文ペアに対して意味的な類似度スコアを予測するためのデータセット。
    #
    # このモデルは、2つの文を入力として受け取り、
    # その文ペアがどの程度意味的に近いかを score として出力する。
    model="llm-book/bert-base-japanese-v3-jsts",
    # function_to_apply="none" は、
    # モデルが出力した値に sigmoid や softmax などの変換をかけず、
    # 生のスコアをそのまま返す指定。
    #
    # 分類タスクでは通常、出力logitにsoftmaxをかけて確率に変換することが多い。
    # しかし、文類似度のような回帰タスクでは、
    # モデルの出力値そのものが類似度スコアとして意味を持つ。
    #
    # そのため、ここでは余計な確率変換を行わないようにしている。
    function_to_apply="none",
)

# 基準となる文を用意する。
#
# この文では、
#   「川べり」
#   「サーフボード」
#   「人たち」
# という情報が含まれている。
#
# 文類似度モデルは、単語の完全一致だけではなく、
# 「サーフボードを持った人たち」と「サーファーたち」のような
# 意味的に近い表現も考慮してスコアを出す。
text = "川べりでサーフボードを持った人たちがいます"

# text と意味的に近い文を用意する。
#
# 基準文:
#   「川べりでサーフボードを持った人たちがいます」
#
# 比較文:
#   「サーファーたちが川べりに立っています」
#
# 「サーフボードを持った人たち」は、一般に「サーファーたち」と解釈できる。
# また、どちらの文にも「川べり」という場所情報が含まれている。
#
# そのため、この2文は意味的に近く、高めの類似度スコアが出ることが期待される。
sim_text = "サーファーたちが川べりに立っています"

# text と sim_text の類似度を計算する。
#
# pipeline に文ペアを入力するときは、
# {"text": 文1, "text_pair": 文2} の辞書形式で渡す。
#
# 内部では、BERTのTokenizerが2文をまとめて以下のような形に変換する。
#
#   [CLS] 文1 [SEP] 文2 [SEP]
#
# BERTはTransformer Encoderを使って、
# 2つの文の単語同士の関係をSelf-Attentionで捉える。
#
# Self-Attentionでは、各トークンが他のトークンを参照しながら、
# 文脈を考慮した表現ベクトルに変換される。
#
# 文ペア全体の表現は、主に [CLS] トークンに対応するベクトルとして取り出され、
# その後の回帰用の出力層によって類似度スコアに変換される。
result = text_sim_pipeline({"text": text, "text_pair": sim_text})

# result は辞書形式で返る。
#
# 例:
#   {"score": 4.1}
#
# のように、score に文ペアの意味的類似度が格納される。
#
# モデルやデータセットによってスコア範囲は異なるが、
# JSTS系のモデルでは、一般にスコアが大きいほど意味的に似ていると解釈する。
print(result["score"])

# text と意味的に遠い文を用意する。
#
# 基準文:
#   「川べりでサーフボードを持った人たちがいます」
#
# 比較文:
#   「トイレの壁に黒いタオルがかけられています」
#
# 比較文には、
#   「トイレ」
#   「壁」
#   「黒いタオル」
# という情報が含まれている。
#
# これは基準文の
#   「川べり」
#   「サーフボード」
#   「人たち」
# とは意味的にほとんど対応しない。
#
# そのため、この2文は意味的に遠く、低い類似度スコアが出ることが期待される。
dissim_text = "トイレの壁に黒いタオルがかけられています"

# text と dissim_text の類似度を計算する。
#
# ここでも同じモデルを使うが、
# 入力される文ペアの意味的な重なりが少ないため、
# sim_text と比較した場合よりも低い score になることが期待される。
#
# このように、同じ基準文に対して複数の比較文を入力すると、
# どの文が意味的に近いかをランキングする用途にも使える。
result = text_sim_pipeline({"text": text, "text_pair": dissim_text})

# 類似度スコアを表示する。
#
# スコアが高い:
#   2つの文が意味的に似ている
#
# スコアが低い:
#   2つの文が意味的に異なる
#
# ただし、このスコアはモデルが学習データから獲得した統計的な判断であり、
# 人間の直感と完全に一致するとは限らない。
print(result["score"])


Loading weights: 100%|██████████| 201/201 [00:00<00:00, 22904.13it/s]


2.1525161266326904
0.8183554410934448


In [6]:
from transformers import pipeline
from torch.nn.functional import cosine_similarity

# Hugging Face Transformers の pipeline を使って、
# 文を「ベクトル表現」に変換する。
#
# ここでは、2文の類似度を直接スコアとして出す STS モデルではなく、
# まず各文を埋め込みベクトル embedding に変換し、
# その後でベクトル同士のコサイン類似度を計算する。
#
# この方法は、文検索・類似文検索・クラスタリング・RAG の検索部分などでよく使われる。
#
# 例:
#   文A → ベクトルA
#   文B → ベクトルB
#   ベクトルAとベクトルBの向きが近いほど、意味的に近い文だとみなす
#
# コサイン類似度は、2つのベクトルの「向きの近さ」を測る指標。
# ベクトルの長さではなく、方向の近さを見るため、
# 文埋め込みの類似度計算でよく使われる。
#
# Math:
# \cos(\theta) = \frac{\mathbf{a} \cdot \mathbf{b}}{\|\mathbf{a}\|\|\mathbf{b}\|}
#
# 値の目安:
#   1 に近い   → ベクトルの向きが近い、つまり意味が近い
#   0 に近い   → 関係が弱い
#   -1 に近い  → ベクトルの向きが反対
sim_enc_pipeline = pipeline(
    # llm-book/bert-base-japanese-v3-unsup-simcse-jawiki は、
    # 日本語BERTをベースにした SimCSE 系の文埋め込みモデル。
    #
    # SimCSE は Simple Contrastive Learning of Sentence Embeddings の略。
    # 文を意味的なベクトル空間に写像するための手法。
    #
    # 通常のBERTは、分類や穴埋めには強い一方で、
    # そのまま文ベクトルとして使うと類似度計算に最適化されていない場合がある。
    #
    # SimCSEでは、対照学習 contrastive learning によって、
    # 意味的に近い文のベクトルは近く、
    # 意味的に遠い文のベクトルは離れるように学習する。
    #
    # unsup は教師なし学習を意味する。
    # 教師なしSimCSEでは、同じ文をDropoutなどの揺らぎを通して2回エンコードし、
    # それらを正例ペアとして扱う。
    #
    # jawiki は、日本語Wikipediaなどの日本語コーパスを用いて学習されていることを示す。
    model="llm-book/bert-base-japanese-v3-unsup-simcse-jawiki",
    # task="feature-extraction" を指定すると、
    # pipeline は分類ラベルではなく、モデル内部の隠れ状態ベクトルを返す。
    #
    # つまり、
    #   入力文 → トークン列 → BERT → 各トークンのベクトル
    # という形で、特徴量を取り出す。
    #
    # この特徴量を使って、後でコサイン類似度を計算する。
    task="feature-extraction",
)

# 類似度を調べたい基準文を用意する。
#
# この文は、
#   「川べり」
#   「サーフボード」
#   「人たち」
# という意味情報を含んでいる。
text = "川べりでサーフボードを持った人たちがいます"

# text と意味的に近い比較文を用意する。
#
# 「サーフボードを持った人たち」と「サーファーたち」は意味的に近く、
# さらに「川べり」という場所情報も一致している。
#
# そのため、text と sim_text の文ベクトルは近い向きになることが期待される。
sim_text = "サーファーたちが川べりに立っています"

# text と意味的に遠い比較文を用意する。
#
# 「トイレ」「壁」「黒いタオル」は、
# text に含まれる「川べり」「サーフボード」「人たち」と対応しにくい。
#
# そのため、text と dissim_text の文ベクトルはあまり近くならないことが期待される。
dissim_text = "トイレの壁に黒いタオルがかけられています"

# text のベクトル表現を獲得する。
#
# sim_enc_pipeline(text, return_tensors=True) を実行すると、
# PyTorch Tensor 形式で特徴量が返る。
#
# 戻り値の形は、おおよそ以下のようになる。
#
#   [batch_size, sequence_length, hidden_size]
#
# batch_size:
#   入力した文の数。
#   今回は1文だけなので 1。
#
# sequence_length:
#   トークン数。
#   BERTでは文がサブワード単位に分割されるため、
#   日本語の文字数や単語数と完全には一致しない。
#
# hidden_size:
#   各トークンを表すベクトルの次元数。
#   BERT-base 系では多くの場合 768 次元。
#
# [0] は batch の先頭、つまり今回入力した1文を取り出す。
# さらに [0] は、その文の先頭トークンに対応するベクトルを取り出す。
#
# BERTでは先頭に [CLS] トークンが置かれることが多く、
# [CLS] ベクトルは文全体の意味を集約した表現として使われる。
#
# つまり text_emb は、text 全体を表す文ベクトルとして扱っている。
text_emb = sim_enc_pipeline(text, return_tensors=True)[0][0]

# sim_text のベクトル表現を獲得する。
#
# text_emb と同じモデルでエンコードすることで、
# 2つの文を同じベクトル空間上の点として比較できる。
#
# 異なるモデルで作ったベクトル同士は空間の意味が揃わないため、
# 通常は同じエンコーダで埋め込みを作る必要がある。
sim_emb = sim_enc_pipeline(sim_text, return_tensors=True)[0][0]

# text と sim_text の類似度を計算する。
#
# cosine_similarity(text_emb, sim_emb, dim=0) は、
# 2つのベクトルについて0次元方向、つまりベクトル全体に沿って
# コサイン類似度を計算する。
#
# text_emb と sim_emb はどちらも hidden_size 次元の1次元ベクトルなので、
# dim=0 を指定している。
#
# 意味的に近い文であれば、
# 埋め込みベクトルの向きも近くなり、
# コサイン類似度は高くなることが期待される。
sim_pair_score = cosine_similarity(text_emb, sim_emb, dim=0)

# .item() は、PyTorch Tensor に入っている単一の数値を
# Python の float として取り出すためのメソッド。
#
# print すると、text と sim_text の意味的な近さが数値として表示される。
print(sim_pair_score.item())

# dissim_text のベクトル表現を獲得する。
#
# text や sim_text と同じ SimCSE エンコーダを使うことで、
# dissim_text も同じ意味ベクトル空間に配置される。
dissim_emb = sim_enc_pipeline(dissim_text, return_tensors=True)[0][0]

# text と dissim_text の類似度を計算する。
#
# dissim_text は text と意味的に離れているため、
# sim_pair_score よりも低い値になることが期待される。
#
# このように、文を一度ベクトル化しておくと、
# 多数の候補文との類似度を計算して、
# 「基準文に意味的に近い文」をランキングできる。
#
# RAG Retrieval-Augmented Generation では、
# ユーザーの質問文をベクトル化し、
# 文書データベース内のベクトルと比較して、
# 類似度の高い文書を検索する。
# そのため、このコードはRAGの検索処理の基礎にもなっている。
dissim_pair_score = cosine_similarity(text_emb, dissim_emb, dim=0)

# text と dissim_text のコサイン類似度を表示する。
#
# 期待される結果としては、
#   sim_pair_score    の方が高い
#   dissim_pair_score の方が低い
# となる。
#
# ただし、モデルの学習データやトークナイザの性質によって、
# 必ず人間の直感通りの値になるとは限らない。
print(dissim_pair_score.item())


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 25792.36it/s]


0.8568587899208069
0.4588705003261566


In [7]:
from pprint import pprint
from transformers import pipeline

# Hugging Face Transformers の pipeline を使って、
# 日本語文から固有表現 Named Entity を抽出する。
#
# 固有表現抽出は NER Named Entity Recognition と呼ばれるタスク。
# 文章中から、人名・地名・組織名・作品名・日付など、
# 特定の意味カテゴリを持つ表現を検出する。
#
# 例:
#   「大谷翔平は岩手県出身です」
#
# という文があった場合、
#   大谷翔平 → 人名
#   岩手県   → 地名
# のように、文字列の範囲とカテゴリを推定する。
#
# NER は、検索エンジン、質問応答、情報抽出、RAGの前処理、
# ニュース記事解析、人物・組織・地名の自動抽出などで使われる。
ner_pipeline = pipeline(
    # llm-book/bert-base-japanese-v3-ner-wikipedia-dataset は、
    # 日本語BERTを固有表現抽出タスク向けにファインチューニングしたモデル。
    #
    # BERT は Transformer Encoder に基づく言語モデル。
    # Self-Attention により、各トークンが文中の他のトークンを参照しながら、
    # 文脈を考慮した表現ベクトルに変換される。
    #
    # NERでは、文章全体を入力し、各トークンに対して
    # 「このトークンは人名の一部か」
    # 「このトークンは地名の一部か」
    # 「このトークンは固有表現ではないか」
    # のようなラベルを予測する。
    #
    # つまり、文章全体に1つのラベルを付けるテキスト分類とは異なり、
    # NERはトークン単位の系列ラベリング sequence labeling タスクである。
    model="llm-book/bert-base-japanese-v3-ner-wikipedia-dataset",
    # aggregation_strategy="simple" は、
    # サブワード単位・トークン単位で出力された予測結果を、
    # 人間が読みやすい固有表現単位にまとめる指定。
    #
    # BERT系モデルでは、入力文が単語単位ではなく、
    # WordPieceやSentencePieceなどのサブワード単位に分割されることがある。
    #
    # 例えば「大谷翔平」という語が、
    # 内部的には複数のトークンに分かれて処理される可能性がある。
    #
    # aggregation_strategy を指定しない場合、
    # それぞれのトークンごとに別々の結果が返ることがあり、
    # 出力が細かくなりすぎる。
    #
    # "simple" を指定すると、連続する同種のラベルをまとめて、
    # 「大谷翔平」全体を1つの人名エンティティとして扱いやすくなる。
    aggregation_strategy="simple",
)

# 固有表現を抽出したい日本語文を用意する。
#
# この文には、複数の固有表現が含まれている。
#
#   大谷翔平 → 人名
#   岩手県   → 地名・地域名
#   水沢市   → 地名・地域名
#
# また、「プロ野球選手」は職業・一般名詞であり、
# モデルのラベル体系によっては固有表現として扱われない可能性がある。
#
# NERモデルは、単語そのものだけではなく、
# 文脈も考慮してラベルを判断する。
# 例えば「水沢市」は地名として使われているため、
# Location系のラベルとして抽出されることが期待される。
text = "大谷翔平は岩手県水沢市出身のプロ野球選手"

# text 中の固有表現を抽出する。
#
# ner_pipeline(text) を実行すると、内部では主に以下の処理が行われる。
#
# 1. 入力文をトークン化する
# 2. 各トークンをIDに変換する
# 3. BERTに入力し、各トークンの文脈表現を得る
# 4. 各トークンに対して固有表現ラベルを予測する
# 5. aggregation_strategy に従って連続トークンをまとめる
# 6. 固有表現の文字列・ラベル・スコア・位置情報を返す
#
# 出力には、一般に以下のような情報が含まれる。
#
# entity_group:
#   固有表現の種類。
#   例: 人名、地名、組織名など。
#
# score:
#   モデルがそのラベルだと判断した信頼度。
#   1に近いほど自信が高い。
#
# word:
#   抽出された固有表現の文字列。
#
# start:
#   入力文中で、その固有表現が始まる文字位置。
#
# end:
#   入力文中で、その固有表現が終わる文字位置。
#
# NERでは、start / end の位置情報が重要。
# これにより、元の文章中のどこに固有表現があるかを特定できる。
pprint(ner_pipeline(text))


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 35191.27it/s]


[{'end': None,
  'entity_group': '人名',
  'score': np.float32(0.99823624),
  'start': None,
  'word': '大谷 翔平'},
 {'end': None,
  'entity_group': '地名',
  'score': np.float32(0.9986874),
  'start': None,
  'word': '岩手 県 水沢 市'}]


In [8]:
from transformers import pipeline

# Hugging Face Transformers の pipeline を使って、
# 日本語の記事本文から要約文を生成する。
#
# ここで使うタスクは text2text-generation である。
# text2text-generation は、
# 「入力もテキスト、出力もテキスト」という形式の生成タスクを扱う。
#
# 代表的な例として、以下のようなタスクがある。
#
# 入力テキスト → 出力テキスト
#
# 長い記事本文 → 要約文
# 日本語文 → 英語翻訳
# 質問文 → 回答文
# 誤字を含む文 → 修正文
#
# このように、入力と出力をどちらもテキストとして統一的に扱うのが
# T5 系モデルの大きな特徴である。
text2text_pipeline = pipeline(
    # llm-book/t5-base-long-livedoor-news-corpus は、
    # T5 をベースに、日本語ニュース記事の要約タスク向けに
    # ファインチューニングされたモデルである。
    #
    # livedoor-news-corpus は、日本語ニュース記事を含むデータセットで、
    # 記事本文とタイトル・要約などを用いた日本語自然言語処理の実験でよく使われる。
    #
    # T5 は Text-to-Text Transfer Transformer の略で、
    # 翻訳・要約・分類・質問応答などをすべて
    # 「テキストを入力してテキストを出力する問題」として扱う。
    #
    # BERT が主に Encoder のみを使うモデルであるのに対して、
    # T5 は Encoder-Decoder 型の Transformer である。
    #
    # Encoder:
    #   入力文全体を読み取り、文脈を考慮した表現に変換する。
    #
    # Decoder:
    #   Encoder の出力を参照しながら、要約文を1トークンずつ生成する。
    #
    # 要約タスクでは、
    # Encoder が記事全体の内容を理解し、
    # Decoder が重要な情報を短い文章として生成する。
    model="llm-book/t5-base-long-livedoor-news-corpus"
)

# 要約したい記事本文を用意する。
#
# この article には、NHKスペシャルで取り上げられる
# スティーブ・ジョブズ氏に関する番組紹介文が入っている。
#
# 内容としては、以下の情報が含まれている。
#
# 1. 3連休にテレビを見る人へ番組をすすめている
# 2. NHKスペシャル「世界を変えた男 スティーブ・ジョブズ」が紹介されている
# 3. ジョブズ氏の生い立ちやアップル社からの追放経験に触れている
# 4. ジョブズ氏が追い求めた未来について番組で扱うと述べている
# 5. ジョブズ氏の伝記が日本でもベストセラーになっている
#
# 要約モデルは、これらの情報のうち重要度が高い部分を抽出・圧縮し、
# 短い要約文を生成する。
article = "ついに始まった３連休。テレビを見ながら過ごしている人も多いのではないだろうか？　今夜オススメなのは何と言っても、NHKスペシャル「世界を変えた男 スティーブ・ジョブズ」だ。実は知らない人も多いジョブズ氏の養子に出された生い立ちや、アップル社から一時追放されるなどの経験。そして、彼が追い求めた理想の未来とはなんだったのか、ファンならずとも気になる内容になっている。 今年、亡くなったジョブズ氏の伝記は日本でもベストセラーになっている。今後もアップル製品だけでなく、世界でのジョブズ氏の影響は大きいだろうと想像される。ジョブズ氏のことをあまり知らないという人もこの機会にぜひチェックしてみよう。 世界を変えた男　スティーブ・ジョブズ（NHKスペシャル）"

# article の要約を生成する。
#
# text2text_pipeline(article) を実行すると、
# 内部では主に以下の処理が行われる。
#
# 1. 入力記事をトークン化する
#    日本語文をモデルが扱えるトークン列に分割し、ID列に変換する。
#
# 2. Encoder に入力する
#    Encoder は Self-Attention によって、
#    記事中の各トークンが他のトークンとどのような関係にあるかを考慮する。
#
# 3. Decoder が要約文を生成する
#    Decoder は、すでに生成したトークンと Encoder の出力を参照しながら、
#    次に出すべきトークンを1つずつ予測する。
#
# 4. 生成されたトークン列を文字列に戻す
#    モデルが出力したトークンID列を自然な日本語文に変換する。
#
# text2text_pipeline(article) の戻り値は、通常リスト形式で返る。
#
# 例:
# [
#     {
#         "generated_text": "生成された要約文"
#     }
# ]
#
# そのため、[0] で最初の生成結果を取り出し、
# ["generated_text"] で要約文だけを取り出している。
print(text2text_pipeline(article)[0]["generated_text"])


Loading weights: 100%|██████████| 284/284 [00:00<00:00, 28599.13it/s]
[transformers] The model 'T5ForConditionalGeneration' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM', 'DeepseekV4ForCausalLM', 'DiffLlamaForCausalLM', 'DogeForCausalLM', 'Dots1ForCausalLM', 'ElectraForCausalLM', 'Emu3ForCausalLM', 'ErnieForCausalLM', 'Ernie4_5ForCausalLM', 'Ernie4_5_MoeForCausalL

ついに始まった３連休。テレビを見ながら過ごしている人も多いのではないだろうか？　今夜オススメなのは何と言っても、NHKスペシャル「世界を変えた男 スティーブ・ジョブズ」だ。実は知らない人も多いジョブズ氏の養子に出された生い立ちや、アップル社から一時追放されるなどの経験。そして、彼が追い求めた理想の未来とはなんだったのか、ファンならずとも気になる内容になっている。 今年、亡くなったジョブズ氏の伝記は日本でもベストセラーになっている。今後もアップル製品だけでなく、世界でのジョブズ氏の影響は大きいだろうと想像される。ジョブズ氏のことをあまり知らないという人もこの機会にぜひチェックしてみよう。 世界を変えた男　スティーブ・ジョブズ（NHKスペシャル）
